<a href="https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA1/IL1.1/1-github_model_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA1/IL1.1/1-github_model_api.ipynb)


## Primera Llamada al Modelo

En este ejercicio, aprenderemos a realizar nuestra primera llamada a un modelo de lenguaje usando la API de **Mistral**.

# 1. Conexión Directa con el Cliente OpenAI

## Objetivos de Aprendizaje
- Configurar una conexión directa con Mistral usando el cliente OpenAI
- Comprender los parámetros básicos de configuración de API
- Implementar llamadas básicas a modelos de lenguaje
- Aplicar mejores prácticas de seguridad con API keys

## Introducción
Mistral da acceso gratuito a varios modelos de lenguaje mediante una **API compatible con OpenAI**.
Eso significa que usamos la misma librería `openai` de siempre: solo cambiamos el `base_url`.
En este notebook aprenderemos a:
1. Configurar el entorno y las credenciales
2. Establecer una conexión con la API
3. Realizar llamadas básicas al modelo
4. Explorar diferentes parámetros de configuración

## Configuración de Credenciales

- **En Google Colab:** carga tus keys en el panel 🔑 **Secrets** de la barra lateral
  (`LLM_API_KEY`) y activa "Notebook access". La celda de abajo las lee sola.
- **En local:** copia `.env.example` a `.env` en la raíz del repo y complétalo.

Tu key gratuita de Mistral se obtiene en [console.mistral.ai/api-keys](https://console.mistral.ai/api-keys).

**Mejores Prácticas de Seguridad:**
- Nunca hardcodees API keys en el código
- Usa los Secrets de Colab o un archivo `.env`
- No compartas credenciales en repositorios públicos
- Rota las API keys regularmente


In [ ]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q openai python-dotenv


In [ ]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

# Importar las bibliotecas necesarias
from openai import OpenAI
import os

# Verificar que tenemos las bibliotecas correctas
print("OpenAI library version:", __import__('openai').__version__)
print("Python version:", __import__('sys').version)

# Configuración del cliente: la librería `openai` apuntada a Groq
try:
    # Configurar el cliente con variables de entorno
    client = OpenAI(
        base_url=os.environ.get("LLM_BASE_URL"),
        api_key=os.environ.get("LLM_API_KEY")
    )

    # Verificar configuración (sin mostrar la API key completa por seguridad)
    print("Base URL configurada:", client.base_url)
    print("API Key configurada:", "✓" if client.api_key else "✗")

    if client.api_key:
        # NUNCA imprimas la key, ni siquiera un fragmento: el output queda guardado
        # dentro del .ipynb y viaja al repositorio cuando haces commit.
        print(f"API Key cargada correctamente ({len(client.api_key)} caracteres)")
    else:
        print("⚠️  API Key no encontrada. Asegúrate de configurar LLM_API_KEY")

except Exception as e:
    print(f"Error en configuración: {e}")
    print("Verifica que las variables de entorno estén configuradas correctamente")

OpenAI library version: 2.54.0
Python version: 3.13.13 (main, Apr  7 2026, 18:19:01) [Clang 21.0.0 (clang-2100.0.123.102)]
Base URL configurada: https://api.mistral.ai/v1/
API Key configurada: ✓
API Key cargada correctamente (32 caracteres)


In [ ]:
# Primera llamada básica al modelo
def llamada_basica():
    try:
        response = client.chat.completions.create(
            model=os.getenv("LLM_MODEL", "mistral-small-latest"),
            messages=[
                {"role": "user", "content": "Hola, ¿cómo estás? Responde en una oración."}
            ],
            temperature=0.1,
            max_tokens=150
        )

        print("=== Respuesta del Modelo ===")
        print(response.choices[0].message.content)
        print("\n=== Información Técnica ===")
        print(f"Modelo usado: {response.model}")
        print(f"Tokens usados: {response.usage.total_tokens}")
        print(f"Tokens de entrada: {response.usage.prompt_tokens}")
        print(f"Tokens de salida: {response.usage.completion_tokens}")

    except Exception as e:
        print(f"Error en la llamada: {e}")
        print("Verifica tu configuración y conexión a internet")

# Ejecutar la función
llamada_basica()

=== Respuesta del Modelo ===
¡Hola! Estoy aquí para ayudarte, así que estoy muy bien, gracias por preguntar. 😊

=== Información Técnica ===
Modelo usado: mistral-small-latest
Tokens usados: 56
Tokens de entrada: 30
Tokens de salida: 26


## Usando Roles del Sistema

El rol "system" permite establecer el comportamiento y contexto del asistente antes de la conversación.

In [ ]:
# Ejemplo con mensaje de sistema
def usar_mensaje_sistema():
    try:
        response = client.chat.completions.create(
            model=os.getenv("LLM_MODEL", "mistral-small-latest"),
            messages=[
                {
                    "role": "system",
                    "content": "Eres un experto en tecnología que explica conceptos complejos de manera simple y amigable. Siempre incluyes ejemplos prácticos."
                },
                {
                    "role": "user",
                    "content": "¿Qué es una API?"
                }
            ],
            temperature=0.7,
            max_tokens=200
        )

        print("=== Respuesta con Mensaje de Sistema ===")
        print(response.choices[0].message.content)

    except Exception as e:
        print(f"Error: {e}")

# Ejecutar función
usar_mensaje_sistema()

=== Respuesta con Mensaje de Sistema ===
¡Claro! Imagina que una **API (Interfaz de Programación de Aplicaciones)** es como un **menú de restaurante**.

### 📜 **¿Qué hace una API?**
Una API es un **intermediario** que permite que dos sistemas diferentes hablen entre sí sin necesidad de conocer los detalles internos del otro. Es como un camarero que toma tu pedido (tu solicitud) y lo lleva a la cocina (el sistema), y luego te trae la comida (la respuesta).

---

### 🍔 **Ejemplo práctico: Pedir comida a domicilio**
1. **Tú (Cliente)**: Quieres pedir una pizza.
   - No entras a la cocina ni sabes cómo se hace, solo pides lo que quieres.

2. **API (Camarero)**: Toma tu pedido y lo lleva a la cocina.
   - La API es el puente entre tu app (ej: Uber Eats) y el restaurante


## Explorando Parámetros de Configuración

Los parámetros más importantes al hacer llamadas a LLMs son:

- **temperature**: Controla la creatividad (0.0 = determinístico, 1.0 = muy creativo)
- **max_tokens**: Límite de tokens en la respuesta
- **model**: El modelo específico a usar (`mistral-small-latest`, `ministral-8b-latest`, etc.)
- **messages**: Array de mensajes con roles (system, user, assistant)

In [ ]:
# Comparando diferentes valores de temperature
def comparar_temperature():
    prompt = "Escribe una historia muy corta sobre un robot que aprende a cocinar."

    temperatures = [0.1, 0.5, 0.9]

    for temp in temperatures:
        print(f"\n{'='*50}")
        print(f"TEMPERATURE: {temp}")
        print('='*50)

        try:
            response = client.chat.completions.create(
                model=os.getenv("LLM_MODEL", "mistral-small-latest"),
                messages=[{"role": "user", "content": prompt}],
                temperature=temp,
                max_tokens=100
            )

            print(response.choices[0].message.content)
            print(f"\nTokens usados: {response.usage.total_tokens}")

        except Exception as e:
            print(f"Error: {e}")

# Ejecutar comparación
comparar_temperature()


TEMPERATURE: 0.1


**"El Chef de Chatarra"**

El robot **C-7** fue diseñado para reparar naves espaciales, pero su mayor sueño era cocinar. Cada vez que los humanos dejaban restos de comida cerca de su panel de carga, sus sensores captaban los aromas y su CPU se llenaba de curiosidad.

Un día, decidió experimentar. Tomó un huevo del almacén, lo rompió contra una sartén (que encontró en la basura

Tokens usados: 131

TEMPERATURE: 0.5


**"El Chef de Acero"**

El robot **C-37** fue diseñado para limpiar laboratorios, pero un día, al escuchar risas desde la cocina, se coló por curiosidad.

Observó cómo la chef **Mara** mezclaba harina, huevos y azúcar con movimientos precisos. Sus sensores captaron cada gesto: el *crack* de los huevos, el *susurro* de la batidora, el *silencio

Tokens usados: 131

TEMPERATURE: 0.9


**"El Chef de Acero"**

El robot *C-47* despertó con un zumbido de curiosidad. Su última actualización pedía "aprender cocina humana". Con sus dedos metálicos, analizó una receta de tortilla de patatas en su base de datos.

—¡Iniciando proceso de cocción —anunció, mientras cortaba las patatas con precisión quirúrgica.

La sartén, al contacto con el aceite,

Tokens usados: 131


## Ejercicios Prácticos

### Ejercicio 1: Experimentar con Diferentes Modelos
Modifica el código para probar diferentes modelos disponibles (si tienes acceso):
- `mistral-small-latest` (el que usamos por defecto)
- `ministral-8b-latest` (más rápido y liviano)
- `openai/gpt-oss-20b`

Revisa todos los modelos disponibles en la [documentación de Mistral](https://docs.mistral.ai/getting-started/models/models_overview/).

> **Ojo con los modelos de razonamiento** (como `openai/gpt-oss-120b`): consumen parte del
> presupuesto de `max_tokens` en razonamiento interno que no ves, así que con valores bajos
> pueden devolver una respuesta vacía. Es un buen experimento para el Ejercicio 3.

### Ejercicio 2: Crear un Asistente Especializado
Diseña un mensaje de sistema para crear un asistente especializado en un tema específico (ejemplo: finanzas, salud, educación).

### Ejercicio 3: Optimización de Tokens
Experimenta con diferentes valores de max_tokens para encontrar el equilibrio entre respuesta completa y eficiencia de costos.

## Conceptos Clave

1. **Configuración segura** de APIs usando variables de entorno
2. **Parámetros básicos** para controlar el comportamiento del modelo
3. **Manejo de errores** en llamadas a APIs
4. **Roles de mensajes** (system, user, assistant)
5. **Monitoreo de uso** de tokens y costos

## Próximos Pasos

En el siguiente notebook exploraremos cómo LangChain simplifica y abstrae estas operaciones, proporcionando herramientas más poderosas para el desarrollo de aplicaciones con LLMs.